# 数据获取模块：中国金融与宏观数据

本文件是课程的**通用数据层**，为第4章和第5章提供真实数据支持。
所有数据获取函数封装后可在各章直接调用。

## 数据来源优先级
| 优先级 | 包 | 特点 | 安装 |
|--------|-----|------|------|
| 1 | `akshare` | 免费、无需注册、覆盖最广 | `pip install akshare` |
| 2 | `baostock` | 免费、无需注册、历史数据稳定 | `pip install baostock` |
| 3 | `tushare` | 需积分、数据质量高 | `pip install tushare` |

## 离线备份说明
每个数据模块提供两个代码块：
- **🌐 在线版**：直接从数据源获取，需要网络
- **💾 离线版**：从本地 `data/` 目录读取预先下载的 CSV

建议课前运行在线版保存数据，课堂演示时使用离线版以防网络不稳定。

---
**目录**
- [0. 安装与全局设置](#sec0)
- [1. 个股日行情数据](#sec1)
- [2. 指数日行情数据](#sec2)
- [3. K 线数据（baostock）](#sec3)
- [4. 宏观数据：CPI 与 GDP](#sec4)
- [5. 数据保存汇总](#sec5)


---
<a id='sec0'></a>
## 0. 安装与全局设置

In [ ]:
# 安装所需包（只需运行一次）
# !pip install akshare baostock pandas numpy matplotlib seaborn -q

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from pathlib import Path

# ── 中文字体 ──────────────────────────────────────────────
plt.rcParams['font.sans-serif'] = ['PingFang SC']    # macOS
# plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']  # Windows
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi']  = 120
plt.rcParams['font.size']   = 11
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

# ── 离线数据目录 ──────────────────────────────────────────
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
print(f'✅ 全局设置完成，离线数据目录：{DATA_DIR.resolve()}')

---
<a id='sec1'></a>
## 1. 个股日行情数据

**用途**：第5章（收益率分布）、第4章 Section 4（多股票相关性）

**股票池**：贵州茅台、比亚迪、宁德时代、招商银行、中国平安（覆盖消费、新能源、金融三个行业）

In [ ]:
# 股票池配置（修改这里即可更换股票）
STOCKS = {
    '贵州茅台': '600519',
    '比亚迪':   '002594',
    '宁德时代': '300750',
    '招商银行': '600036',
    '中国平安': '601318',
}

START_DATE = '20200101'
END_DATE   = '20241231'

print('股票池：')
for name, code in STOCKS.items():
    market = 'sh' if code.startswith('6') else 'sz'
    print(f'  {name}：{market}{code}')

### 🌐 在线版：通过 akshare 获取

In [ ]:
import akshare as ak

def fetch_stock_akshare(name, code, start, end):
    """获取单只股票日行情，返回标准化 DataFrame"""
    market = 'sh' if code.startswith('6') else 'sz'
    symbol = market + code
    try:
        df = ak.stock_zh_a_hist(
            symbol=code,
            period='daily',
            start_date=start,
            end_date=end,
            adjust='qfq'   # 前复权
        )
        df = df[['日期', '收盘', '成交量']].copy()
        df.columns = ['date', 'close', 'volume']
        df['date'] = pd.to_datetime(df['date'])
        df = df.set_index('date').sort_index()
        df['stock'] = name
        # 计算日对数收益率（%）
        df['log_return'] = np.log(df['close'] / df['close'].shift(1)) * 100
        return df.dropna()
    except Exception as e:
        print(f'  ⚠️  {name}（{code}）获取失败：{e}')
        return None

# 批量获取
print('正在获取个股日行情数据...')
stock_dfs = {}
for name, code in STOCKS.items():
    df = fetch_stock_akshare(name, code, START_DATE, END_DATE)
    if df is not None:
        stock_dfs[name] = df
        print(f'  ✅ {name}：{len(df)} 条记录 '
              f'（{df.index[0].date()} ~ {df.index[-1].date()}）')

# 合并为宽表（收盘价）和长表（收益率）
close_wide = pd.DataFrame({name: df['close']
                           for name, df in stock_dfs.items()})
ret_wide   = pd.DataFrame({name: df['log_return']
                           for name, df in stock_dfs.items()})
ret_long   = ret_wide.stack().reset_index()
ret_long.columns = ['date', 'stock', 'log_return']

print(f'\n收盘价宽表：{close_wide.shape}，收益率宽表：{ret_wide.shape}')
print(ret_wide.describe().round(4))

### 💾 离线版：从本地 CSV 读取

In [ ]:
# 读取离线数据（课前已保存好的 CSV）
close_wide = pd.read_csv(DATA_DIR / 'stock_close_wide.csv',
                         index_col='date', parse_dates=True)
ret_wide   = pd.read_csv(DATA_DIR / 'stock_return_wide.csv',
                         index_col='date', parse_dates=True)
ret_long   = pd.read_csv(DATA_DIR / 'stock_return_long.csv',
                         parse_dates=['date'])

print(f'✅ 离线数据加载成功')
print(f'收盘价宽表：{close_wide.shape}')
print(f'收益率宽表：{ret_wide.shape}')
print(ret_wide.describe().round(4))

### 📊 快速预览：五只股票的收益率分布

In [ ]:
from scipy import stats

# ── 描述统计 ──────────────────────────────────────────────
desc = ret_wide.agg([
    'mean', 'std',
    lambda x: stats.skew(x.dropna()),
    lambda x: stats.kurtosis(x.dropna())
]).T
desc.columns = ['均值(%)', '标准差(%)', '偏度', '超额峰态']
print('── 五只股票日对数收益率描述统计 ──')
print(desc.round(4))

# ── 分组 KDE 图 ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
palette = sns.color_palette('colorblind', len(ret_wide.columns))

# 左图：叠加 KDE
for (name, series), color in zip(ret_wide.items(), palette):
    sns.kdeplot(series.dropna(), ax=axes[0], color=color,
                linewidth=1.8, label=name)
# 叠加正态密度参考线
x = np.linspace(-12, 12, 400)
axes[0].plot(x, stats.norm.pdf(x, 0, ret_wide.stack().std()),
             'k--', linewidth=1.2, alpha=0.5, label='正态参考')
axes[0].set_xlim(-12, 12)
axes[0].set_xlabel('日对数收益率（%）', fontsize=10)
axes[0].set_ylabel('密度', fontsize=10)
axes[0].set_title('五只股票日收益率分布\n（均呈厚尾特征，偏离正态分布）',
                  fontsize=11, fontweight='bold', loc='left')
axes[0].legend(fontsize=9)
sns.despine(ax=axes[0])

# 右图：分组箱线图
sns.boxplot(data=ret_long, x='stock', y='log_return',
            order=list(ret_wide.columns),
            palette=palette, width=0.5, ax=axes[1],
            flierprops=dict(marker='.', markersize=2, alpha=0.3))
axes[1].axhline(0, color='gray', linewidth=0.8, linestyle='--')
axes[1].set_xlabel('')
axes[1].set_ylabel('日对数收益率（%）', fontsize=10)
axes[1].set_title('五只股票收益率波动对比\n（宁德时代、比亚迪波动率显著更高）',
                  fontsize=11, fontweight='bold', loc='left')
axes[1].tick_params(axis='x', rotation=15)
sns.despine(ax=axes[1])

plt.suptitle(f'A股五只代表性股票日收益率（{START_DATE[:4]}–{END_DATE[:4]}，前复权）',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
<a id='sec2'></a>
## 2. 指数日行情数据

**用途**：第4章 Section 5（时间序列可视化）

**指数池**：上证综指、沪深300、创业板指、中证500

In [ ]:
# 指数配置
INDICES = {
    '上证综指': 'sh000001',
    '沪深300':  'sh000300',
    '创业板指': 'sz399006',
    '中证500':  'sh000905',
}
IDX_START = '20150101'
IDX_END   = '20241231'

### 🌐 在线版：通过 akshare 获取

In [ ]:
import akshare as ak

def fetch_index_akshare(name, symbol, start, end):
    """获取指数日行情"""
    try:
        df = ak.stock_zh_index_daily(symbol=symbol)
        df['date'] = pd.to_datetime(df['date'])
        df = df.set_index('date').sort_index()
        df = df.loc[start:end, ['close']].copy()
        df.columns = [name]
        return df
    except Exception as e:
        print(f'  ⚠️  {name} 获取失败：{e}')
        return None

print('正在获取指数日行情...')
idx_dfs = []
for name, symbol in INDICES.items():
    df = fetch_index_akshare(name, symbol, IDX_START, IDX_END)
    if df is not None:
        idx_dfs.append(df)
        print(f'  ✅ {name}：{len(df)} 条')

index_close = pd.concat(idx_dfs, axis=1).dropna(how='all')

# 计算归一化指数（基期=100，用于多指数走势对比）
index_norm = index_close.div(index_close.iloc[0]) * 100

# 计算日对数收益率
index_ret = np.log(index_close / index_close.shift(1)) * 100

print(f'\n指数收盘价宽表：{index_close.shape}')
print(index_close.tail(3))

### 💾 离线版：从本地 CSV 读取

In [ ]:
index_close = pd.read_csv(DATA_DIR / 'index_close.csv',
                          index_col='date', parse_dates=True)
index_norm  = index_close.div(index_close.iloc[0]) * 100
index_ret   = np.log(index_close / index_close.shift(1)) * 100

print(f'✅ 指数数据加载成功，形状：{index_close.shape}')
print(index_close.tail(3))

### 📊 快速预览：四大指数走势与收益率分布

In [ ]:
palette = sns.color_palette('colorblind', len(index_close.columns))
linestyles = ['-', '--', '-.', ':']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8))

# ── 上图：归一化指数走势（折线图）────────────────────────
for (name, series), color, ls in zip(index_norm.items(), palette, linestyles):
    ax1.plot(series.index, series, color=color, linewidth=1.6,
             linestyle=ls, label=name)

# 重大事件标注
events = [
    ('2015-06-12', '2015\n股灾'),
    ('2018-01-26', '2018\n贸易战'),
    ('2020-01-23', '2020\n新冠'),
    ('2022-10-31', '2022\n政策底'),
]
for date_str, label in events:
    d = pd.Timestamp(date_str)
    if d in index_norm.index or d > index_norm.index[0]:
        ax1.axvline(d, color='gray', linewidth=0.8,
                    linestyle=':', alpha=0.6)
        ax1.text(d, ax1.get_ylim()[1] * 0.97, label,
                 ha='center', fontsize=8, color='gray', va='top')

ax1.axhline(100, color='black', linewidth=0.6, linestyle='--', alpha=0.4)
ax1.set_ylabel('归一化指数（基期=100）', fontsize=10)
ax1.set_xlabel('')
ax1.set_title(f'四大 A 股指数走势对比（{IDX_START[:4]}–{IDX_END[:4]}，基期=100）\n'
              '创业板指弹性最大，上证综指最为稳健',
              fontsize=12, fontweight='bold', loc='left')
ax1.legend(fontsize=10, loc='upper left')
ax1.xaxis.set_major_formatter(mpl.dates.DateFormatter('%Y'))
sns.despine(ax=ax1)

# ── 下图：指数日收益率分布（分组箱线图，按年）────────────
# 取沪深300为例，展示年度分布变化
hs300_ret = index_ret['沪深300'].dropna().to_frame()
hs300_ret['year'] = hs300_ret.index.year

yearly_data = [hs300_ret[hs300_ret['year'] == yr]['沪深300'].values
               for yr in sorted(hs300_ret['year'].unique())]
years = sorted(hs300_ret['year'].unique())

bp = ax2.boxplot(yearly_data, labels=years,
                 patch_artist=True, widths=0.6,
                 medianprops=dict(color='#d7191c', linewidth=2),
                 flierprops=dict(marker='.', markersize=2, alpha=0.3))

# 按年标准差着色（波动越大颜色越深）
yearly_std = [np.std(d) for d in yearly_data]
norm_std = plt.Normalize(min(yearly_std), max(yearly_std))
cmap = plt.cm.Blues
for patch, std_val in zip(bp['boxes'], yearly_std):
    patch.set_facecolor(cmap(norm_std(std_val)))

ax2.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.4)
ax2.set_xlabel('')
ax2.set_ylabel('日对数收益率（%）', fontsize=10)
ax2.set_title('沪深300年度日收益率分布\n（箱体颜色深浅代表波动率高低）',
              fontsize=12, fontweight='bold', loc='left')
ax2.tick_params(axis='x', labelsize=9)
sns.despine(ax=ax2)

plt.tight_layout()
plt.show()

---
<a id='sec3'></a>
## 3. K 线数据

**用途**：第4章 Section 5.5（K线图）

baostock 历史数据稳定，适合 K 线图；akshare 提供备用方案。

In [ ]:
# K 线配置
KLINE_STOCK  = '600519'  # 贵州茅台
KLINE_NAME   = '贵州茅台'
KLINE_START  = '2024-01-01'
KLINE_END    = '2024-12-31'

### 🌐 在线版 A：通过 baostock 获取（推荐，数据稳定）

In [ ]:
import baostock as bs

def fetch_kline_baostock(code, start, end, name='股票'):
    """通过 baostock 获取日 K 线数据（OHLCV）"""
    lg = bs.login()
    if lg.error_code != '0':
        print(f'baostock 登录失败：{lg.error_msg}')
        return None

    market = 'sh' if code.startswith('6') else 'sz'
    symbol = f'{market}.{code}'

    rs = bs.query_history_k_data_plus(
        symbol,
        'date,open,high,low,close,volume,amount',
        start_date=start,
        end_date=end,
        frequency='d',
        adjustflag='2'   # 前复权
    )
    rows = []
    while rs.error_code == '0' and rs.next():
        rows.append(rs.get_row_data())
    bs.logout()

    df = pd.DataFrame(rows, columns=rs.fields)
    for col in ['open', 'high', 'low', 'close', 'volume', 'amount']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date').sort_index().dropna()
    print(f'✅ {name}（{symbol}）K线数据：{len(df)} 条 '
          f'（{df.index[0].date()} ~ {df.index[-1].date()}）')
    return df

kline_df = fetch_kline_baostock(KLINE_STOCK, KLINE_START, KLINE_END, KLINE_NAME)
print(kline_df.head(3))

### 🌐 在线版 B：通过 akshare 获取（备用）

In [ ]:
import akshare as ak

def fetch_kline_akshare(code, start, end, name='股票'):
    """通过 akshare 获取日 K 线数据（OHLCV）"""
    start_str = start.replace('-', '')
    end_str   = end.replace('-', '')
    try:
        df = ak.stock_zh_a_hist(
            symbol=code, period='daily',
            start_date=start_str, end_date=end_str,
            adjust='qfq'
        )
        df = df[['日期','开盘','最高','最低','收盘','成交量','成交额']].copy()
        df.columns = ['date','open','high','low','close','volume','amount']
        df['date'] = pd.to_datetime(df['date'])
        df = df.set_index('date').sort_index()
        print(f'✅ {name} K线数据：{len(df)} 条')
        return df
    except Exception as e:
        print(f'⚠️  获取失败：{e}')
        return None

kline_df = fetch_kline_akshare(KLINE_STOCK, KLINE_START, KLINE_END, KLINE_NAME)
print(kline_df.head(3))

### 💾 离线版：从本地 CSV 读取

In [ ]:
kline_df = pd.read_csv(DATA_DIR / f'kline_{KLINE_STOCK}.csv',
                       index_col='date', parse_dates=True)
print(f'✅ K线数据加载成功：{len(kline_df)} 条')
print(kline_df.head(3))

### 📊 快速预览：K 线图（mplfinance）

In [ ]:
# !pip install mplfinance -q
import mplfinance as mpf

# 取最近 60 个交易日
df_plot = kline_df.tail(60).copy()

# 计算 20 日和 60 日移动均线
df_plot['MA20'] = df_plot['close'].rolling(20).mean()
df_plot['MA60'] = kline_df['close'].rolling(60).mean().reindex(df_plot.index)

# 添加均线到图形
add_plots = [
    mpf.make_addplot(df_plot['MA20'], color='#2166ac',
                     linewidths=1.2, label='MA20'),
    mpf.make_addplot(df_plot['MA60'], color='#d7191c',
                     linewidths=1.2, label='MA60'),
]

# 自定义样式（去除默认黑色背景）
mc = mpf.make_marketcolors(
    up='#d7191c', down='#2166ac',  # 中国惯例：红涨蓝跌
    edge='inherit',
    wick='inherit',
    volume='inherit'
)
style = mpf.make_mpf_style(
    marketcolors=mc,
    gridstyle=':',
    gridcolor='#e0e0e0',
    facecolor='white',
    edgecolor='white',
    rc={'font.sans-serif': ['PingFang SC'],
        'axes.unicode_minus': False}
)

fig, axlist = mpf.plot(
    df_plot,
    type='candle',
    style=style,
    addplot=add_plots,
    volume=True,
    title=f'\n{KLINE_NAME} 日 K 线图（近 60 个交易日）',
    ylabel='价格（元）',
    ylabel_lower='成交量（手）',
    figsize=(12, 7),
    returnfig=True
)

# 手动添加图例
axlist[0].legend(['MA20', 'MA60'], loc='upper left', fontsize=9)
plt.show()

---
<a id='sec4'></a>
## 4. 宏观数据：CPI 与 GDP

**用途**：第4章 Section 5（时间序列可视化）、第5章（分布演变）

### 🌐 在线版：通过 akshare 获取

In [ ]:
import akshare as ak

# ── CPI 月度数据 ──────────────────────────────────────────
def fetch_cpi_akshare():
    """获取中国 CPI 月度同比数据"""
    try:
        df = ak.macro_china_cpi_monthly()
        # akshare 字段名可能随版本变化，做容错处理
        df.columns = [c.strip() for c in df.columns]
        # 尝试识别日期列和 CPI 同比列
        date_col = [c for c in df.columns if '日期' in c or 'date' in c.lower()][0]
        yoy_col  = [c for c in df.columns if '同比' in c][0]
        df = df[[date_col, yoy_col]].copy()
        df.columns = ['date', 'cpi_yoy']
        df['date'] = pd.to_datetime(df['date'])
        df['cpi_yoy'] = pd.to_numeric(df['cpi_yoy'], errors='coerce')
        df = df.set_index('date').sort_index().dropna()
        print(f'✅ CPI 月度数据：{len(df)} 条 '
              f'（{df.index[0].date()} ~ {df.index[-1].date()}）')
        return df
    except Exception as e:
        print(f'⚠️  CPI 获取失败：{e}')
        print('   字段列表：', df.columns.tolist() if 'df' in dir() else 'N/A')
        return None

# ── GDP 季度数据 ──────────────────────────────────────────
def fetch_gdp_akshare():
    """获取中国 GDP 季度同比数据"""
    try:
        df = ak.macro_china_gdp()
        df.columns = [c.strip() for c in df.columns]
        date_col = [c for c in df.columns if '季度' in c or '时间' in c or 'date' in c.lower()][0]
        yoy_col  = [c for c in df.columns if '同比' in c][0]
        df = df[[date_col, yoy_col]].copy()
        df.columns = ['date', 'gdp_yoy']
        df['date'] = pd.to_datetime(df['date'])
        df['gdp_yoy'] = pd.to_numeric(df['gdp_yoy'], errors='coerce')
        df = df.set_index('date').sort_index().dropna()
        print(f'✅ GDP 季度数据：{len(df)} 条 '
              f'（{df.index[0].date()} ~ {df.index[-1].date()}）')
        return df
    except Exception as e:
        print(f'⚠️  GDP 获取失败：{e}')
        return None

cpi_df = fetch_cpi_akshare()
gdp_df = fetch_gdp_akshare()

### 💾 离线版：从本地 CSV 读取

In [ ]:
cpi_df = pd.read_csv(DATA_DIR / 'macro_cpi.csv',
                     index_col='date', parse_dates=True)
gdp_df = pd.read_csv(DATA_DIR / 'macro_gdp.csv',
                     index_col='date', parse_dates=True)

print(f'✅ CPI 数据：{len(cpi_df)} 条')
print(f'✅ GDP 数据：{len(gdp_df)} 条')

### 📊 快速预览：CPI 与 GDP 时序走势

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=False)

# ── 上图：CPI 同比 ────────────────────────────────────────
ax1.plot(cpi_df.index, cpi_df['cpi_yoy'],
         color='#d7191c', linewidth=1.5)
ax1.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.4)
ax1.axhline(3, color='#d7191c', linewidth=0.8, linestyle=':',
            alpha=0.5, label='3% 调控目标')
ax1.fill_between(cpi_df.index, cpi_df['cpi_yoy'], 0,
                 where=(cpi_df['cpi_yoy'] > 3),
                 color='#d7191c', alpha=0.15, label='高通胀区间')
ax1.fill_between(cpi_df.index, cpi_df['cpi_yoy'], 0,
                 where=(cpi_df['cpi_yoy'] < 0),
                 color='#2166ac', alpha=0.15, label='通缩区间')
ax1.set_ylabel('CPI 同比（%）', fontsize=10)
ax1.set_title('中国 CPI 月度同比：近年通胀整体温和，2023年后呈通缩压力',
              fontsize=11, fontweight='bold', loc='left')
ax1.legend(fontsize=9, loc='upper right')
ax1.xaxis.set_major_formatter(mpl.dates.DateFormatter('%Y'))
sns.despine(ax=ax1)

# ── 下图：GDP 季度同比 ────────────────────────────────────
colors_gdp = ['#d7191c' if v >= 0 else '#2166ac'
              for v in gdp_df['gdp_yoy']]
ax2.bar(gdp_df.index, gdp_df['gdp_yoy'],
        color=colors_gdp, width=80, alpha=0.8)
ax2.axhline(0, color='black', linewidth=0.8)

# 标注新冠冲击
if pd.Timestamp('2020-01-01') in gdp_df.index or \
   gdp_df.index.min() < pd.Timestamp('2020-06-01'):
    ax2.axvspan(pd.Timestamp('2020-01-01'),
                pd.Timestamp('2020-06-30'),
                alpha=0.1, color='gray', label='新冠冲击')
    ax2.text(pd.Timestamp('2020-04-01'), gdp_df['gdp_yoy'].min() * 0.9,
             '新冠冲击', ha='center', fontsize=8, color='gray')

ax2.set_ylabel('GDP 同比增速（%）', fontsize=10)
ax2.set_title('中国 GDP 季度同比增速：新冠冲击后复苏，近年增速趋稳',
              fontsize=11, fontweight='bold', loc='left')
ax2.xaxis.set_major_formatter(mpl.dates.DateFormatter('%Y'))
sns.despine(ax=ax2)

plt.tight_layout()
plt.show()

---
<a id='sec5'></a>
## 5. 数据保存汇总

运行完在线版代码块后，执行以下代码将所有数据保存为 CSV，
供后续课堂演示时离线使用。

In [ ]:
# ── 保存个股数据 ──────────────────────────────────────────
if 'close_wide' in dir() and close_wide is not None:
    close_wide.to_csv(DATA_DIR / 'stock_close_wide.csv')
    ret_wide.to_csv(DATA_DIR / 'stock_return_wide.csv')
    ret_long.to_csv(DATA_DIR / 'stock_return_long.csv', index=False)
    print('✅ 个股数据已保存（3 个文件）')

# ── 保存指数数据 ──────────────────────────────────────────
if 'index_close' in dir() and index_close is not None:
    index_close.to_csv(DATA_DIR / 'index_close.csv')
    print('✅ 指数数据已保存')

# ── 保存 K 线数据 ─────────────────────────────────────────
if 'kline_df' in dir() and kline_df is not None:
    kline_df.to_csv(DATA_DIR / f'kline_{KLINE_STOCK}.csv')
    print(f'✅ K线数据已保存：kline_{KLINE_STOCK}.csv')

# ── 保存宏观数据 ──────────────────────────────────────────
if 'cpi_df' in dir() and cpi_df is not None:
    cpi_df.to_csv(DATA_DIR / 'macro_cpi.csv')
    print('✅ CPI 数据已保存')
if 'gdp_df' in dir() and gdp_df is not None:
    gdp_df.to_csv(DATA_DIR / 'macro_gdp.csv')
    print('✅ GDP 数据已保存')

# ── 汇总报告 ──────────────────────────────────────────────
print(f'\n── data/ 目录文件清单 ──')
for f in sorted(DATA_DIR.glob('*.csv')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:35s}  {size_kb:6.1f} KB')

---
## 附录：akshare 接口稳定性说明

akshare 接口会随数据源变化而更新，偶尔出现字段名变更或接口失效。
遇到报错时，建议：

1. 升级到最新版本：`pip install akshare --upgrade`
2. 查看官方文档：https://akshare.akfamily.xyz/
3. 切换备用方案（baostock 或离线 CSV）

**常见报错与解决方法**

| 报错 | 原因 | 解决 |
|------|------|------|
| `KeyError: '日期'` | 字段名变更 | `print(df.columns)` 查看实际字段名 |
| `ConnectionError` | 网络问题 | 切换离线版或检查代理设置 |
| `JSONDecodeError` | 数据源暂时不可用 | 稍等几分钟后重试 |
| `No data returned` | 股票代码格式错误 | 检查是否需要加市场前缀（sh/sz）|
